In [0]:
%sql
use catalog formula1

In [0]:
from pyspark.sql.functions import col,isnan,trim,isnan,current_timestamp
from pyspark.sql.types import IntegerType, StringType, DoubleType, DateType, TimestampType,NumericType


In [0]:
df = spark.read.format("delta") \
    .load("/Volumes/formula1/bronze_001/formula1_vol/raw_data_formula1")

In [0]:
df = df.drop("_c0")
df = df.drop("positionText")
df = df.drop("positionOrder")
df = df.drop("fp1_date")
df = df.drop("fp1_time")
df = df.drop("fp2_date")
df = df.drop("fp2_time")
df = df.drop("fp3_date")
df = df.drop("fp3_time")
df = df.drop("quali_date")
df = df.drop("quali_time")
df = df.drop("sprint_date")
df = df.drop("sprint_time")
df = df.drop("positionText_driverstandings")
df = df.drop("positionText_constructorstandings")

In [0]:
df.display()


In [0]:
bad_values = ["\\N", "/N", "//N", "None", "NULL", "", " "]
for colname in df.columns:
    df = df.replace(bad_values, None, subset=[colname])

int_columns = ["resultId","raceId", "driverId", "constructorId", "number", "grid", "position", "points", "laps", "rank", "statusId", "year", "round", "circuitId",  "lap","number_drivers","constructorStandingsId","position_constructorstandings","wins_constructorstandings","driverStandingsId", "points_driverstandings", "position_driverstandings","wins","points_constructorstandings"]
date_columns = ["date", "dob"]
string_col = ["time", "fastestLapTime", "fastestLapSpeed", "name_x", "time_races", "url_x", "circuitRef", "name_y", "location", "country",  "alt", "url_y", "driverRef", "code", "forename", "surname", "nationality", "url", "constructorRef", "name", "nationality_constructors", "url_constructors", "time_laptimes", "time_pitstops", "status","milliseconds", "fastestLap","position_laptimes", "milliseconds_laptimes"
       ,"stop", "lap_pitstops", "milliseconds_pitstops" ,"duration"]
double_col = ["lat", "lng"]

for c in int_columns:
    if c in df.columns:
        df = df.withColumn(c, col(c).cast(IntegerType()))
for c in date_columns:
    if c in df.columns:
        df = df.withColumn(c, col(c).cast(DateType()))
for c in string_col:
    if c in df.columns:
        df = df.withColumn(c, col(c).cast(StringType()))
for c in double_col:
    if c in df.columns:
        df = df.withColumn(c, col(c).cast(DoubleType()))

display(df)


In [0]:
df = df.withColumnRenamed('name_x', 'race_name') \
    .withColumnRenamed('date', 'race_date') \
    .withColumnRenamed('time_races', 'race_time') \
    .withColumnRenamed('url_x', 'race_url') \
    .withColumnRenamed("constructorRef","constructor_ref") \
    .withColumnRenamed("name_y","circuit_name") \
    .withColumnRenamed("driverRef", "driver_ref") \
    .withColumnRenamed("forename", "firstname") \
    .withColumnRenamed("surname", "lastname") \
    .withColumnRenamed("dob", "date_of_birth") \
    .withColumnRenamed("circuitRef","circuit_ref") \
    .withColumnRenamed("name_y","circuit_ref") \
    .withColumnRenamed("url_y","url_circuit")

In [0]:
df.write.mode("append").format("delta").option("mergeSchema", "true").saveAsTable("formula1.silver_001.f1_silver_vol_cleandataset")

df.display()